[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/orientation/optical_rotation/optical_rotation.ipynb)

# Optical Rotation, Out and Back

Quartz and a magnetic field both turn the polarization of light passing through them. When the light is sent back through the same medium, they part ways: quartz turns it back, while the field keeps turning it, which is how an optical isolator stops reflections from returning to a laser.

The difference is orientation. The light's electric field is modelled as a bivector of spacetime, extrinsic, and a polarizer reads it through the plane its axis sweeps in time, `field & (axis & time)`. Quartz turns the light in the plane its direction of travel sweeps in time, `direction & time`, an intrinsic bivector that lies across the path as a turn; the quartz's rotary power is an intrinsic scalar, a sense of its own, and only their product is an extrinsic rate that exponentiates, so the turn reverses with the direction of travel. The magnetic field, a plane of space, turns the light by its flux through the beam's cross-section, `magnetic_field & cross_section`, which is the same whichever way the light goes. Mirroring every input of the setup mirrors every result, with no sign rule added.

In Jones calculus the polarization is a Jones vector and each medium a rotation matrix, $R(\rho L)$ for quartz of specific rotation $\rho$ and length $L$, and $R(VBL)$ for a Faraday rotator of Verdet constant $V$ in a field of strength $B$. In the lab's fixed frame the round trip through quartz is $R(-\rho L)\,R(\rho L) = 1$, and through the rotator $R(VBL)\,R(VBL) = R(2VBL)$, a quarter turn when $VBL = 45^\circ$.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
import numpy as np

from numga import NumpyContext, stack
from numga.algebras import STA
from examples.util.animation import save_figure
from examples.orientation.optical_rotation import render

In [ ]:
# Spacetime, with space positive, `x+y+z+t-`.
ga = STA
mv = NumpyContext(ga).multivector
# + extrinsic, - intrinsic
Scalar = ga.gatype.scalar()                       # Scalar+
# Directions, of travel and of a polarizer's axis, are antivectors.
Direction = ga.gatype.antivector()                # Antivector-
# Electric and magnetic fields alike are bivectors.
Field = ga.gatype.bivector()                      # Bivector+
TurnRate = ga.gatype.bivector()                   # Bivector+
# The beam's cross-section, the patch two directions across the path span.
CrossSection = Field.derive.flipped_orientation   # Bivector-
# A turn per distance with a sense of its own: measured against a handedness.
RotaryPower = Scalar.derive.flipped_orientation   # Scalar-

# The direction of time of an observer at rest with the media, which pairs to one with `mv.t`; the dual
# carries the minus of t's square.
time = -mv.t.dual()                               # [] Direction-
# The field along a direction, the dual of the plane the direction sweeps in time: the vacuum's
# identification of a direction with the field along it, pairing to one with that plane for a unit direction.
field_along = (Direction & time).dual()           # [] Field+ <- Direction-

Light travels along `direction`, and its electric field is the field along its polarization, `field_along(polarization)`, a bivector of spacetime. An optically active medium turns the field across the path by its rotary power per distance. The light sweeps a plane through spacetime, its direction joined with time, `direction & time`: an intrinsic bivector oriented by the travel, which as a turn lies across the path. An intrinsic bivector is not a rate, and `exp` refuses it. The rotary power is what makes it one: the medium's turn has a sense of its own, measured against a handedness, so the rotary power is an intrinsic scalar, and the product of the two is an extrinsic bivector that exponentiates.

A magnetic field, a plane of space, turns the light by its flux through the beam's cross-section, `magnetic_field & cross_section`, the patch the two directions across the path span, times the Verdet constant per distance and field; the turn lies in the cross-section's plane, `cross_section.dual_inverse()`. Neither the flux nor the cross-section knows which way the light goes, so the field turns the returning light the same way as the outgoing light. Of the light that comes back, the polarizer it entered through passes the share `(field & (axis & time)).squared()`, by Malus's law.

In [ ]:
# The path through the medium in mm, and the samples of the light along each leg.
length = 2.0
n_samples = 9
depths = np.linspace(0, length, n_samples)
# Quartz turns 21.7 degrees per mm; the rotary power in rad per mm.
quartz_rotary_power = np.deg2rad(22.5)
# The Verdet constant in rad per mm per tesla, illustrative, and the magnetic field's strength in tesla.
verdet = np.deg2rad(22.5)
field_strength = 1.0

direction = mv.z.dual()                                                         # [] Direction-
polarizer_axis = mv.x.dual()                                                    # [] Direction-
cross_section = mv.xy.dual()                                                    # [] CrossSection-
entering = field_along(polarizer_axis)                                          # [] Field+
# Measured against the lab's handedness, the basis volume x ^ y ^ z.
rotary_power = mv(RotaryPower, [quartz_rotary_power])                           # [] RotaryPower-
# A plane of space, across the path.
magnetic_field = mv.xy * field_strength                                         # [] Field+


def out_and_back(out_rate: TurnRate, back_rate: TurnRate, entering: Field) -> Field:
    # Out to a reflector at the far end and back, [legs, depths]: the light leaves where it entered.
    out = (out_rate * (-depths / 2)).exp() >> entering                          # [depths] Field+
    back = (back_rate * (-depths / 2)).exp() >> out[-1]                         # [depths] Field+
    return stack([out, back])                                                   # [legs, depths] Field+


def through_media(direction: Direction, cross_section: CrossSection, rotary_power: RotaryPower,
                  magnetic_field: Field, entering: Field) -> Field:
    # [media, legs, depths]: the optically active medium, then the magnetic field.
    # The plane the light sweeps through time, out and back.
    sweep, back_sweep = direction & time, -direction & time                     # [] Bivector-, Bivector-
    active = out_and_back(sweep * rotary_power, back_sweep * rotary_power, entering)
    # The magnetic field's flux through the cross-section, turning the light in the cross-section's plane.
    faraday_rate = cross_section.dual_inverse() * (magnetic_field & cross_section) * verdet   # [] TurnRate+
    faraday = out_and_back(faraday_rate, faraday_rate, entering)
    return stack([active, faraday])                                             # [media, legs, depths] Field+


original = through_media(direction, cross_section, rotary_power, magnetic_field, entering)   # [media, legs, depths] Field+
# Malus's law: the share of the returning light that passes the polarizer it entered through.
passed = (original[:, 1, -1] & (polarizer_axis & time)).squared()               # [media] Scalar+

In [ ]:
print("Share of the returning light that passes the polarizer it entered through")
for medium, share in zip(("Quartz", "Magnetic field"), passed):
    print(medium)
    print(share.coefficient_table())

The mirror image of the setup reflects the beam, the medium, the magnetic field and the incoming light alike. The rotary power, an intrinsic scalar, changes sign: the mirror image of right-handed quartz is left-handed quartz. The magnetic field, a plane of space, reflects as a plane does. Each mirrored setup then gives the mirror image of the original light, at every depth on both legs.

The figure unfolds each path along the beam, out to the reflector and back, with quartz on the left and the magnetic field on the right, the original setups above their mirror images. Each line is the polarization across the beam, the direction of the light's field, orange on the way out and teal on the way back; the dashed lines at both ends are the polarization the light entered with.

In [ ]:
# The plane x = 0, holding the path.
mirror = mv.x                                                                   # [] Plane+
mirrored = through_media(mirror >> direction, mirror >> cross_section, mirror >> rotary_power,
                         mirror >> magnetic_field, mirror >> entering)          # [media, legs, depths] Field+

fields = stack([original, mirrored], axis=1)                                    # [media, setups, legs, depths] Field+
save_figure(render.draw_isolator(fields, depths), "optical_rotation")

In [ ]:
# checks
# The quartz returns the light as it entered; the magnetic field returns it crossed with the polarizer's axis,
# so the polarizer stops it. The mirrored setups give the mirror image of the original light, at every depth
# on both legs.
active, faraday = original[:, 1, -1]
mismatch = (mirrored - (mirror >> original)).scalar_norm_squared()              # [media, legs, depths] Scalar+
assert abs((active - entering).scalar_norm_squared().to_array()) < 1e-24
assert abs((faraday & (polarizer_axis & time)).to_array()) < 1e-12
assert abs(mismatch.to_array()).max() < 1e-24